In [ ]:
# cil imports
from cil.framework import ImageData, ImageGeometry
from cil.framework import AcquisitionGeometry, AcquisitionData

from cil.processors import Slicer, AbsorptionTransmissionConverter, TransmissionAbsorptionConverter

from cil.optimisation.functions import IndicatorBox
from cil.optimisation.algorithms import CGLS, SIRT
from cil.optimisation.utilities import callbacks

from cil.plugins.astra.operators import ProjectionOperator
from cil.plugins.astra.processors import FBP

#from cil.plugins import TomoPhantom
from phantominator import shepp_logan


from cil.utilities import dataexample
from cil.utilities.display import show2D, show1D, show_geometry

# External imports
import numpy as np
import matplotlib.pyplot as plt
import logging


# Make explicit Radon transform

In [ ]:
# number of pixels
n_pixels = 100
n_angles = 90

# Angles
angles = np.linspace(0, 180, n_angles, endpoint=False, dtype=np.float32)


# Setup acquisition geometry
# with sufficient number of projections
ag = AcquisitionGeometry.create_Parallel2D()\
                            .set_angles(angles)\
                            .set_panel(n_pixels, pixel_size=1/n_pixels)

# Setup image geometry
ig = ImageGeometry(voxel_num_x=n_pixels, 
                   voxel_num_y=n_pixels, 
                   voxel_size_x=1/n_pixels, 
                   voxel_size_y=1/n_pixels)

# Get phantom
#phantom = TomoPhantom.get_ImageData(num_model=1, geometry=ig)
phantom = ImageData(np.flip(shepp_logan(n_pixels)), geometry = ig)

In [ ]:
# Create projection operator using Astra-Toolbox.
A = ProjectionOperator(ig, ag, 'cpu')

# Create an acquisition data (numerically)
sino = A.direct(phantom)